# Automated Code Review Board — Demo

**CSE476 — Agentic AI and Intelligent Automation**

This notebook runs the multi-agent pipeline on three sample diffs and shows:
- Which agent ran and which tools it called
- The findings each agent produced
- The Reviewer Manager's merged decision
- The Fix-Suggestion Agent's patch for the top finding
- The tail of the append-only run log

In [ ]:
import os, sys, asyncio, json
import nest_asyncio
nest_asyncio.apply()
from pathlib import Path

# Always use offline mode in the notebook
os.environ['LLM_MODE'] = 'offline'

# Add the repository root to sys.path so 'code_review_board' can be imported
cwd = Path('.').resolve()
# If we are in code_review_board directory, repo root is parent; otherwise it's cwd
project_root = cwd.parent if cwd.name == 'code_review_board' else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(cwd) not in sys.path:
    sys.path.insert(0, str(cwd))

from code_review_board.orchestrator import run_pipeline
from code_review_board.state.change_state import Severity
from code_review_board.agents.logger import LOG_PATH

SAMPLES = project_root / 'code_review_board' / 'samples' if (project_root / 'code_review_board' / 'samples').exists() else cwd / 'samples'
REPO_ROOT = str(project_root / 'code_review_board' if (project_root / 'code_review_board').exists() else cwd)
print(f'Project root : {project_root}')
print(f'Samples dir  : {SAMPLES}')
print(f'Log file     : {LOG_PATH}')

Project root : D:\Sem 7\Agentic_AI_CA2
Samples dir  : D:\Sem 7\Agentic_AI_CA2\code_review_board\samples
Log file     : D:\Sem 7\Agentic_AI_CA2\code_review_board\logs\run_log.jsonl


In [ ]:
# ── Helper: pretty-print a completed ChangeState ─────────────────────────────
SEVERITY_EMOJI = {Severity.HIGH: '🔴', Severity.MEDIUM: '🟡', Severity.LOW: '🔵'}
DECISION_EMOJI = {'APPROVE': '✅', 'REQUEST_CHANGES': '❌', 'CANNOT_REVIEW': '⚠️'}

def print_review(state, label: str):
    print(f'\n{"═"*60}')
    print(f'  SAMPLE: {label}')
    print(f'{"═"*60}')
    print(f'  Run ID       : {state.run_id}')
    print(f'  Diff summary : {state.diff_summary or "(none — error)"}')
    if state.error:
        print(f'  ⚠️  Error      : {state.error}')
    print()

    if state.findings:
        print(f'  Findings ({len(state.findings)}):')
        for f in state.findings:
            emoji = SEVERITY_EMOJI.get(f.severity, '❓')
            print(f'    {emoji} [{f.severity.value:6s}] {f.agent:20s} {f.rule:30s} {f.file}:{f.line}')
            print(f'            {f.message}')
    else:
        print('  Findings: none')

    emoji = DECISION_EMOJI.get(state.decision, '?')
    print(f'\n  {emoji}  Decision : {state.decision}')
    print(f'      Reason  : {state.decision_reason}')

    fix = state.__dict__.get('fix_suggestion')
    if fix:
        print(f'\n  🔧  Fix Suggestion for [{fix["rule"]}] in {fix["file"]}:{fix["line"]}')
        print('  ' + '─'*56)
        for line in fix['patch_text'].splitlines():
            print(f'    {line}')

    if state.agent_errors:
        print(f'\n  ⚠️  Agent errors: {state.agent_errors}')
    print()

## Scenario 1 — Clean diff (expected: APPROVE)

In [ ]:
clean_diff = (SAMPLES / 'clean.diff').read_text()
state1 = asyncio.run(run_pipeline(clean_diff, repo_root=REPO_ROOT))
print_review(state1, 'clean.diff')


════════════════════════════════════════════════════════════
  SAMPLE: clean.diff
════════════════════════════════════════════════════════════
  Run ID       : e12da8e2-49c6-4a8d-8be2-0f75dcce8ca9
  Diff summary : 2 file(s) changed: +32 lines added, -0 lines removed.

  Findings: none

  ✅  Decision : APPROVE
      Reason  : No findings. The change looks clean.



## Scenario 2 — Secret diff (expected: REQUEST_CHANGES, HIGH first)

In [ ]:
secret_diff = (SAMPLES / 'secret.diff').read_text()
state2 = asyncio.run(run_pipeline(secret_diff, repo_root=REPO_ROOT))
print_review(state2, 'secret.diff')


════════════════════════════════════════════════════════════
  SAMPLE: secret.diff
════════════════════════════════════════════════════════════
  Run ID       : 980b5057-405c-4cd6-809f-f9c94c3f07d0
  Diff summary : 1 file(s) changed: +24 lines added, -0 lines removed.

  Findings (5):
    🔴 [HIGH  ] TestCoverageAgent    MISSING_TEST_FILE              src/api_client.py:0
            No test file found for 'src/api_client.py'. Expected: tests/test_api_client.py
    🔴 [HIGH  ] SecurityAgent        SECRET_API_KEY                 src/api_client.py:6
            Possible API key detected on added line.
    🔴 [HIGH  ] SecurityAgent        SECRET_PASSWORD                src/api_client.py:7
            Hardcoded password detected on added line.
    🔴 [HIGH  ] SecurityAgent        UNSAFE_EVAL                    src/api_client.py:18
            Use of `eval()` is a security risk; avoid executing dynamic code.
    🟡 [MEDIUM] SecurityAgent        SUBPROCESS_SHELL               src/api_client.py:24

## Scenario 3 — Untested diff (expected: REQUEST_CHANGES)

In [ ]:
untested_diff = (SAMPLES / 'untested.diff').read_text()
state3 = asyncio.run(run_pipeline(untested_diff, repo_root=REPO_ROOT))
print_review(state3, 'untested.diff')


════════════════════════════════════════════════════════════
  SAMPLE: untested.diff
════════════════════════════════════════════════════════════
  Run ID       : 8b35de60-2576-42a8-bc4e-2d3cfcc1c608
  Diff summary : 1 file(s) changed: +21 lines added, -0 lines removed.

  Findings (4):
    🔴 [HIGH  ] TestCoverageAgent    MISSING_TEST_FILE              src/data_processor.py:0
            No test file found for 'src/data_processor.py'. Expected: tests/test_data_processor.py
    🔵 [LOW   ] StyleAgent           MISSING_DOCSTRING              src/data_processor.py:4
            Function 'normalize_scores' is missing a docstring.
    🔵 [LOW   ] StyleAgent           MISSING_DOCSTRING              src/data_processor.py:9
            Function 'filter_outliers' is missing a docstring.
    🔵 [LOW   ] StyleAgent           MISSING_DOCSTRING              src/data_processor.py:18
            Function 'compute_statistics' is missing a docstring.

  ❌  Decision : REQUEST_CHANGES
      Reason  : Found

## Scenario 4 — Empty diff (expected: CANNOT_REVIEW)

In [ ]:
state4 = asyncio.run(run_pipeline('', repo_root='.'))
print_review(state4, '<empty string>')


════════════════════════════════════════════════════════════
  SAMPLE: <empty string>
════════════════════════════════════════════════════════════
  Run ID       : 2976293c-c487-46eb-b75a-86e5873b2b00
  Diff summary : (none — error)
  ⚠️  Error      : Empty diff — nothing to review.

  Findings: none

  ⚠️  Decision : CANNOT_REVIEW
      Reason  : Pipeline error: Empty diff — nothing to review.



## Run Log — last 20 entries

In [ ]:
print(f'Log file: {LOG_PATH}\n')
if LOG_PATH.exists():
    lines = LOG_PATH.read_text().strip().splitlines()
    print(f'Total entries: {len(lines)}\n')
    print('Last 20 entries:')
    print('─' * 80)
    for line in lines[-20:]:
        entry = json.loads(line)
        ts    = entry['timestamp'][11:19]  # HH:MM:SS
        print(f"[{ts}] {entry['agent']:20s} → {entry['tool']:35s} | {entry['output_summary']}")
else:
    print('Log file not found.')

Log file: D:\Sem 7\Agentic_AI_CA2\code_review_board\logs\run_log.jsonl

Total entries: 489

Last 20 entries:
────────────────────────────────────────────────────────────────────────────────
[17:28:55] ReviewerManager      → decision                            | decision=Decision.REQUEST_CHANGES, reason=Found 4 HIGH-severity finding(s): MISSING_TEST_FILE, SECRET_API_KEY, SECRET_PASS
[17:28:55] FixSuggestionAgent   → get_top_finding                     | top=MISSING_TEST_FILE
[17:28:55] FixSuggestionAgent   → generate_patch                      | patch generated (228 chars)
[17:28:55] DiffAgent            → parse_diff                          | 1 file(s) parsed
[17:28:55] DiffAgent            → summarize_change                    | 1 file(s) changed: +21 lines added, -0 lines removed.
[17:28:55] StyleAgent           → check_line_length_and_naming        | 0 finding(s)
[17:28:55] StyleAgent           → check_docstrings_with_ast           | 3 finding(s)
[17:28:55] SecurityAgent        → sc